# 01 — Exploratory Data Analysis: Olist Brazilian E-commerce

**Purpose:** understand the 9 Olist tables before any SQL is written: row counts, nulls, duplicates, date ranges, outliers, and the joins the master table will need.

**Source:** Brazilian E-Commerce Public Dataset by Olist (Kaggle, CC BY-NC-SA 4.0), downloaded into `data/raw/`. Money is in Brazilian reais (R$). Dates are Sep 2016 – Oct 2018.

**What this notebook decides:** what gets cleaned in `sql/01_clean_orders.sql`, what gets deduplicated in `sql/02_master_table.sql`, and which months the dashboard trend should use. Every finding is also logged in `docs/data_quality_log.md`.

In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', 20)
RAW = '../data/raw'
PROC = '../data/processed'

## 1. Load the 9 tables

In [2]:
orders = pd.read_csv(f'{RAW}/olist_orders_dataset.csv')
customers = pd.read_csv(f'{RAW}/olist_customers_dataset.csv')
order_items = pd.read_csv(f'{RAW}/olist_order_items_dataset.csv')
order_payments = pd.read_csv(f'{RAW}/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(f'{RAW}/olist_order_reviews_dataset.csv')
products = pd.read_csv(f'{RAW}/olist_products_dataset.csv')
sellers = pd.read_csv(f'{RAW}/olist_sellers_dataset.csv')
geolocation = pd.read_csv(f'{RAW}/olist_geolocation_dataset.csv')
translation = pd.read_csv(f'{RAW}/product_category_name_translation.csv')

In [3]:
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                   ('order_payments', order_payments), ('order_reviews', order_reviews),
                   ('products', products), ('sellers', sellers),
                   ('geolocation', geolocation), ('translation', translation)]:
    print(f'{name:16s} {df.shape[0]:>8,} rows x {df.shape[1]} cols')

orders             99,441 rows x 8 cols
customers          99,441 rows x 5 cols
order_items       112,650 rows x 7 cols
order_payments    103,886 rows x 5 cols
order_reviews      99,224 rows x 7 cols
products           32,951 rows x 9 cols
sellers             3,095 rows x 4 cols
geolocation      1,000,163 rows x 5 cols
translation            71 rows x 2 cols


## 2. Orders: status mix and date range

In [4]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [5]:
orders['order_status'].value_counts(normalize=True).round(4)

order_status
delivered      0.9702
shipped        0.0111
canceled       0.0063
unavailable    0.0061
invoiced       0.0032
processing     0.0030
created        0.0001
approved       0.0000
Name: proportion, dtype: float64

In [6]:
orders['order_purchase_timestamp'].min(), orders['order_purchase_timestamp'].max()

('2016-09-04 21:15:19', '2018-10-17 17:30:18')

In [7]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


## 3. Nulls in orders — what is missing, and where it matters

In [8]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Reading: `order_approved_at` is missing for 160 orders, carrier date for 1,783 and customer delivery date for 2,965. Most of the 2,965 are orders that were never delivered, so a missing delivery date is expected there — it is not a value to impute. The exception is checked next.

In [9]:
delivered = orders[orders['order_status'] == 'delivered']
print('delivered orders:', len(delivered))
print('delivered, no customer delivery date:', delivered['order_delivered_customer_date'].isna().sum())

delivered orders: 96478
delivered, no customer delivery date: 8


## 4. Duplicates — full rows, and duplicate keys

In [10]:
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items),
                   ('order_payments', order_payments), ('order_reviews', order_reviews),
                   ('products', products), ('sellers', sellers), ('geolocation', geolocation)]:
    print(f'{name:16s} full-duplicate rows: {df.duplicated().sum():,}')

orders           full-duplicate rows: 0
customers        full-duplicate rows: 0
order_items      full-duplicate rows: 0


order_payments   full-duplicate rows: 0
order_reviews    full-duplicate rows: 0
products         full-duplicate rows: 0
sellers          full-duplicate rows: 0


geolocation      full-duplicate rows: 261,831


In [11]:
print('duplicate order_id in orders:', orders.duplicated(subset=['order_id']).sum())
print('duplicate customer_id in customers:', customers.duplicated(subset=['customer_id']).sum())
print('duplicate (order_id, order_item_id) in items:', order_items.duplicated(subset=['order_id', 'order_item_id']).sum())
print('duplicate (order_id, payment_sequential) in payments:', order_payments.duplicated(subset=['order_id', 'payment_sequential']).sum())

duplicate order_id in orders: 0
duplicate customer_id in customers: 0
duplicate (order_id, order_item_id) in items: 0
duplicate (order_id, payment_sequential) in payments: 0


In [12]:
print('duplicate order_id in reviews:', order_reviews.duplicated(subset=['order_id']).sum())
print('unique orders with a review:', order_reviews['order_id'].nunique(), 'of', len(order_reviews), 'review rows')

duplicate order_id in reviews: 551
unique orders with a review: 98673 of 99224 review rows


Reading: orders, customers, items, payments, products and sellers have no duplicate keys. Reviews have 551 duplicate order_ids — one order, several reviews — so the master table keeps only the latest review per order. Geolocation has 261,831 full-duplicate rows and is keyed by zip prefix, not by order, so it is loaded but never joined into the order-grain master table.

## 5. Payments: types, negatives, and the revenue definition

In [13]:
order_payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [14]:
print('negative payment_value:', (order_payments['payment_value'] < 0).sum())
print('null payment_value:', order_payments['payment_value'].isna().sum())
print('all payments R$:', round(order_payments['payment_value'].sum(), 2))

negative payment_value: 0
null payment_value: 0
all payments R$: 16008872.12


In [15]:
delivered_ids = set(delivered['order_id'])
paid_delivered = order_payments[order_payments['order_id'].isin(delivered_ids)]
paid_other = order_payments[~order_payments['order_id'].isin(delivered_ids)]
print('delivered-order payments R$:', round(paid_delivered['payment_value'].sum(), 2))
print('non-delivered-order payments R$:', round(paid_other['payment_value'].sum(), 2))

delivered-order payments R$: 15422461.77
non-delivered-order payments R$: 586410.35


In [16]:
canceled_ids = set(orders[orders['order_status'] == 'canceled']['order_id'])
print('canceled-order payments R$:', round(order_payments[order_payments['order_id'].isin(canceled_ids)]['payment_value'].sum(), 2))

canceled-order payments R$: 143255.6


Decision: **revenue = payments on delivered orders only (R$ 15,422,461.77)**. The R$ 586,410.35 paid on non-delivered orders is kept in `payment_value_total` for reconciliation but excluded from revenue, otherwise canceled orders would inflate sales. One delivered order (Sep 2016) has no payment row at all, so its revenue is R$ 0 by COALESCE — one order, immaterial, but it explains the R$ 0 in kpi_monthly for 2016-09.

## 6. Products and categories

In [17]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [18]:
print('distinct product categories (Portuguese):', products['product_category_name'].nunique())
print('categories missing from translation:', sorted(set(products['product_category_name'].dropna()) - set(translation['product_category_name'])))

distinct product categories (Portuguese): 73
categories missing from translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


Reading: the same 610 products are missing category, name length, description length and photo count — a block of incomplete catalogue rows, not random missing cells. Two categories (`pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`) have no English translation, so the pipeline falls back to the Portuguese name for them instead of dropping the rows.

## 7. Delivery performance, before any dashboard

In [19]:
d = delivered.copy()
for c in ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    d[c] = pd.to_datetime(d[c])
d['delivery_days'] = (d['order_delivered_customer_date'] - d['order_purchase_timestamp']).dt.total_seconds() / 86400
d['on_time'] = d['order_delivered_customer_date'] <= d['order_estimated_delivery_date']

In [20]:
print('delivered with a delivery date:', d['delivery_days'].notna().sum())
print('on-time rate (dated delivered orders):', round(d['on_time'].mean(), 4))
print(d['delivery_days'].describe().round(2))

delivered with a delivery date: 96470
on-time rate (dated delivered orders): 0.9188


count    96470.00
mean        12.56
std          9.55
min          0.53
25%          6.77
50%         10.22
75%         15.72
max        209.63
Name: delivery_days, dtype: float64


In [21]:
print('negative delivery days:', (d['delivery_days'] < 0).sum())
print('deliveries over 30 days:', (d['delivery_days'] > 30).sum())

negative delivery days: 0
deliveries over 30 days: 4550


Reading: mean delivery is 12.56 days but the median is 10.22 days — a long right tail (max 209.6 days) pulls the mean up. No negative delivery times. On-time rate is computed only over delivered orders with a delivery date; the 8 delivered orders without one are excluded, not counted as late.

## 8. Reviews: the score distribution, and late vs on-time

In [22]:
order_reviews['review_score'].value_counts(normalize=True).sort_index().round(4)

review_score
1    0.1151
2    0.0318
3    0.0824
4    0.1929
5    0.5778
Name: proportion, dtype: float64

In [23]:
r = order_reviews.copy()
r['review_creation_date'] = pd.to_datetime(r['review_creation_date'])
r_latest = r.sort_values('review_creation_date').drop_duplicates('order_id', keep='last')
print('reviews after keeping latest per order:', len(r_latest))

reviews after keeping latest per order: 98673


In [24]:
m = d.merge(r_latest[['order_id', 'review_score']], on='order_id', how='inner')
print(m.groupby('on_time')['review_score'].agg(['mean', 'count']).round(2))

         mean  count
on_time             
False    2.57   7669
True     4.29  88163


In [25]:
late_mean = m.loc[~m['on_time'], 'review_score'].mean()
ontime_mean = m.loc[m['on_time'], 'review_score'].mean()
print(f'on-time {ontime_mean:.2f} vs late {late_mean:.2f} -> late is {(late_mean - ontime_mean) / ontime_mean * 100:.1f}% lower')

on-time 4.29 vs late 2.57 -> late is -40.2% lower


In [26]:
m['bucket'] = pd.cut(m['delivery_days'], bins=[-1, 3, 7, 14, 30, 1000], labels=['0-3d', '4-7d', '8-14d', '15-30d', '30d+'])
m.groupby('bucket', observed=True)['review_score'].agg(['mean', 'count']).round(2)

,mean,count
bucket,,
0-3d,4.48,4733
4-7d,4.40,21200
8-14d,4.31,39997
15-30d,3.97,25464
30d+,2.25,4430


Reading: scores are bimodal — 57.8% are 5 stars, 11.5% are 1 star — so the 4.16 average hides a split. The delivery effect is not gradual: scores hold up to 14 days (4.31), dip at 15–30 days (3.97), then collapse past 30 days (2.25). That cliff, not the average, is the business story.

## 9. Monthly shape and partial months

In [27]:
orders['purchase_month'] = pd.to_datetime(orders['order_purchase_timestamp']).dt.to_period('M')
orders.groupby('purchase_month').size().head(8)

purchase_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
Freq: M, dtype: int64

In [28]:
orders.groupby('purchase_month').size().tail(8)

purchase_month
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

Reading: Sep 2016 has 4 orders, there are no orders in Nov 2016, and Sep–Oct 2018 are cut off mid-month (16 and 4 orders, none delivered). The monthly KPI table keeps all 25 months for reconciliation but flags Jan 2017 – Aug 2018 as `is_full_month`; the dashboard trend uses only full months, otherwise month-over-month growth for Jan 2017 (+6,500% off a R$ 19.62 base) would be nonsense.

## 10. Reconciliation: do the pipeline outputs match this EDA?

In [29]:
kpi = pd.read_csv(f'{PROC}/kpi_monthly.csv')
master = pd.read_csv(f'{PROC}/master_orders.csv')
print('master rows:', len(master), '| orders rows:', len(orders))
print('master revenue (delivered) R$:', round(master['revenue_delivered'].sum(), 2))
print('kpi_monthly revenue R$:', round(kpi['total_revenue'].sum(), 2))
print('kpi months:', len(kpi), '| full months:', int(kpi['is_full_month'].sum()))

master rows: 99441 | orders rows: 99441
master revenue (delivered) R$: 15422461.77
kpi_monthly revenue R$: 15422461.77
kpi months: 25 | full months: 20


In [30]:
print('master on-time rate:', round(master.loc[master['is_delivered'] == 1, 'on_time_flag'].mean(), 4))
print('master avg delivery days:', round(master.loc[master['is_delivered'] == 1, 'delivery_days'].mean(), 2))
print('master avg review (delivered):', round(master.loc[master['is_delivered'] == 1, 'review_score'].mean(), 2))
print('master cancellation rate:', round(master['is_canceled'].mean(), 4))

master on-time rate: 0.9189
master avg delivery days: 12.56
master avg review (delivered): 4.16
master cancellation rate: 0.0063


## Conclusion

The data is usable, but only with four explicit decisions: revenue counts delivered orders only; reviews are deduplicated to the latest per order; delivery KPIs exclude the 8 delivered orders with no delivery date; and the trend uses full months only. Those four decisions are implemented in `sql/01_clean_orders.sql` → `sql/03_kpi_monthly.sql` and defined in `docs/kpi_definitions.md`. The next step is the pipeline itself (`python src/pipeline.py`), then the Tableau dashboard.